In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%cudf.pandas.profile
### cell 20 ###
def add_gap_rows(essay):
    cols = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # filter once on GPU
    df = train[train.id == essay][cols].reset_index(drop=True)
    # compute previous end on GPU
    df['prev_end'] = df.discourse_end.shift(1).fillna(-1).astype('int32')

    # build intermediate gap rows vectorized on GPU
    df_gap = df[df.gap_length > 0]
    gap_rows = df_gap.assign(
        discourse_start = df_gap.prev_end + 1,
        discourse_end   = df_gap.discourse_start - 1,
        discourse_type  = 'Nothing',
        gap_length      = None,
        gap_end_length  = None
    )[cols]

    # build final gap row if needed, filtered on GPU
    last = df.tail(1)
    end_rows = last.assign(
        discourse_start = last.discourse_end + 1,
        discourse_end   = last.discourse_end + 1 + last.gap_end_length,
        discourse_type  = 'Nothing',
        gap_length      = None,
        gap_end_length  = None
    )[cols]
    end_rows = end_rows[last.gap_end_length > 0]

    # concatenate original rows + new gap rows
    new_rows = cudf.concat([gap_rows, end_rows], ignore_index=True)
    df_essay = cudf.concat([df[cols], new_rows], ignore_index=True)
    return df_essay.sort_values('discourse_start').reset_index(drop=True)


def print_colored_essay(essay):
    df_essay = add_gap_rows(essay)
    # bring only needed columns to CPU once
    pd_df = df_essay[['discourse_start', 'discourse_end', 'discourse_type']].to_pandas()
    ents = [
        {'start': int(s), 'end': int(e), 'label': l}
        for s, e, l in zip(
            pd_df.discourse_start, pd_df.discourse_end, pd_df.discourse_type
        )
    ]
    essay_file = (
        f"/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/"
        f"input/feedback-prize-2021/train/{essay}.txt"
    )
    with open(essay_file, 'r') as f:
        data = f.read()
    doc2 = {'text': data, 'ents': ents}
    colors = {
        'Lead': '#EE11D0', 'Position': '#AB4DE1', 'Claim': '#1EDE71',
        'Evidence': '#33FAFA', 'Counterclaim': '#4253C1',
        'Concluding Statement': 'yellow', 'Rebuttal': 'red'
    }
    options = {
        'ents': pd_df.discourse_type.unique().tolist(),
        'colors': colors
    }
    # ... (rest of visualization logic)
